In [1]:
import json
import os
import time
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import inference

In [2]:


print(f"Is CUDA available? {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
print(f"Number of GPUs: {torch.cuda.device_count()}")

if torch.cuda.is_available():
    print(f"Current GPU device index: {torch.cuda.current_device()}")
    print(f"GPU Device Name: {torch.cuda.get_device_name(0)}")


Is CUDA available? True
CUDA version: 13.2
Number of GPUs: 1
Current GPU device index: 0
GPU Device Name: NVIDIA GeForce RTX 3060 Laptop GPU


In [3]:
#model 

from __future__ import annotations

import torch
import torch.nn as nn
import torch.nn.functional as F

import geometry


class Backbone(nn.Module):


    def __init__(self, out_channels: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=5, padding=2), nn.BatchNorm2d(16), nn.ReLU(inplace=True),
            nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, stride=2, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, out_channels, kernel_size=3, stride=2, padding=1), nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class SiamNCCLocalizer(nn.Module):
    def __init__(self, feat_channels: int = 64):
        super().__init__()
        self.backbone = Backbone(out_channels=feat_channels)
        self.head = nn.Sequential(
            nn.Conv2d(feat_channels, 32, kernel_size=1), nn.ReLU(inplace=True),
            nn.Conv2d(32, 1, kernel_size=1),
        )
        self.offset_head = nn.Sequential(
            nn.Conv2d(feat_channels, 32, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(32, 2, kernel_size=1),
        )

import torch
import torch.nn.functional as F


def correlate(self, search_feat: torch.Tensor, ref_feat: torch.Tensor) -> torch.Tensor:
    """
    Replaces BOTH the original groups=b*c conv2d version (slow: cuDNN has no
    fast path for grouped conv with a large kernel and hundreds of groups)
    AND my previous F.unfold version (catastrophic: materializes a tensor of
    size batch*channels*kh*kw*h_out*w_out - for this model's actual shapes
    that's ~131GB, which is why nothing was finishing).

    This computes the same per-channel normalized cross-correlation via an
    explicit shift-and-accumulate loop over the kh*kw=625 kernel offsets.
    Same total FLOPs as either previous version (~16 GFLOPs/forward pass,
    trivial for a GPU), but peak memory stays at roughly
    batch*channels*h_out*w_out (~26M elements, ~105MB) instead of exploding.
    625 sequential ops means some kernel-launch overhead, but that's a
    constant, small cost - nowhere near what you were hitting.
    """
    b, c, h, w = search_feat.shape
    _, _, kh, kw = ref_feat.shape
    h_out = h - kh + 1
    w_out = w - kw + 1

    ref_mean = ref_feat.mean(dim=[2, 3], keepdim=True)
    ref_centered = ref_feat - ref_mean
    ref_var_energy = ref_centered.pow(2).sum(dim=[2, 3], keepdim=True)  # [b,c,1,1]

    local_mean = F.avg_pool2d(search_feat, kernel_size=(kh, kw), stride=1)
    window_sq_mean = F.avg_pool2d(search_feat.pow(2), kernel_size=(kh, kw), stride=1)
    window_var_energy = (window_sq_mean - local_mean.pow(2)).clamp_min(0) * (kh * kw)

    raw_corr = search_feat.new_zeros(b, c, h_out, w_out)
    for i in range(kh):
        for j in range(kw):
            raw_corr += search_feat[:, :, i:i + h_out, j:j + w_out] * ref_centered[:, :, i:i + 1, j:j + 1]

    denom = torch.sqrt(window_var_energy.clamp_min(1e-8)) * torch.sqrt(ref_var_energy.clamp_min(1e-8))
    corr_ncc = raw_corr / (denom + 1e-6)
    return corr_ncc

def forward(self, reference: torch.Tensor, search: torch.Tensor):
    ref_feat = self.backbone(reference)     # [B, C, 25, 25]
    search_feat = self.backbone(search)      # [B, C, 250, 250]
    corr_ncc = self.correlate(search_feat, ref_feat)   # [B, C, 226, 226]

    heatmap = torch.sigmoid(self.head(corr_ncc))        # [B, 1, 226, 226]
    offset_full = self.offset_head(search_feat)          # [B, 2, 250, 250]
    h_out, w_out = heatmap.shape[-2], heatmap.shape[-1]
    _, _, kh, kw = ref_feat.shape
    oy, ox = kh // 2, kw // 2
    offset = offset_full[:, :, oy:oy + h_out, ox:ox + w_out]
    return heatmap, offset

@staticmethod
def valid_grid_size(search_px: int, ref_px: int) -> int:
    return geometry.valid_corr_size(search_px, ref_px)


In [4]:
import math
import torch.nn as nn


def init_heatmap_head_bias(head: nn.Sequential, grid_size: int = 226, prior: float = None):
    """
    Call this once right after constructing SiamNCCLocalizer, e.g.:
        model = SiamNCCLocalizer()
        init_heatmap_head_bias(model.head, grid_size=226)

    Without this, the head's final conv starts at ~0 logit -> sigmoid(0)=0.5
    everywhere, which on a 226x226=51076-pixel grid with exactly 1 positive
    pixel produces a massive initial negative-loss term (this is your
    train_loss=851 at epoch 1). Adam's first few steps then violently drive
    the output toward 0 EVERYWHERE (including at the true location) just to
    kill that huge background loss, and the network gets stuck in that
    "predict background everywhere" local minimum - which matches your
    heatmap_peak values of ~0.03 across the board.

    Standard fix (RetinaNet/CenterNet): initialize the final layer's bias so
    the network STARTS at roughly the true prior probability (~1/grid_size^2)
    instead of 0.5, so the initial loss - and the resulting gradient step -
    is sane instead of enormous.
    """
    if prior is None:
        prior = 1.0 / (grid_size * grid_size)
    prior = min(max(prior, 1e-4), 0.5)  # keep it numerically sane, don't go too extreme
    bias_value = -math.log((1 - prior) / prior)

    final_conv = head[-1]  # nn.Conv2d(32, 1, kernel_size=1) - the logit-producing layer
    assert isinstance(final_conv, nn.Conv2d) and final_conv.out_channels == 1, \
        "expected head[-1] to be the final 1-channel Conv2d logit layer"
    nn.init.constant_(final_conv.bias, bias_value)
    return bias_value

In [5]:
#train

from __future__ import annotations

import os

import cv2
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset

from dataset import WinspPairDataset
from model import SiamNCCLocalizer
import geometry


def focal_loss(pred, target, pos_mask, alpha=2.0, beta=4.0):
    neg_mask = 1.0 - pos_mask
    pos_loss = -torch.log(pred + 1e-6) * (1 - pred) ** alpha * pos_mask
    neg_loss = -torch.log(1 - pred + 1e-6) * pred ** alpha * (1 - target) ** beta * neg_mask
    n_pos = pos_mask.sum().clamp(min=1)
    return (pos_loss.sum() + neg_loss.sum()) / n_pos


def compute_losses(model, batch, device):
    ref, search = batch["reference"].to(device), batch["search"].to(device)
    hm_t = batch["heatmap"].to(device)
    off_t = batch["offset"].to(device)
    off_m = batch["offset_mask"].to(device)

    hm_p, off_p = model(ref, search)
    loss_hm = focal_loss(hm_p, hm_t, off_m)
    loss_off = (torch.abs(off_p - off_t) * off_m).sum() / off_m.sum().clamp(min=1)
    return loss_hm + loss_off, loss_hm.item(), loss_off.item()


def train(train_dir, val_dir, epochs=40, batch_size=16, lr=1e-3, device="cuda", out_path="driftsense_v2.pt"):
    device = torch.device(device if torch.cuda.is_available() else "cpu")

    train_dl = DataLoader(WinspPairDataset(train_dir), batch_size=batch_size, shuffle=True, num_workers=4)
    val_dl = DataLoader(WinspPairDataset(val_dir), batch_size=batch_size, shuffle=False, num_workers=2)

    model = SiamNCCLocalizer().to(device)
    init_heatmap_head_bias(model.head, grid_size=226)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    best_val = float("inf")
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for batch in train_dl:
            loss, _, _ = compute_losses(model, batch, device)
            opt.zero_grad()
            loss.backward()
            opt.step()
            train_loss += loss.item()
        sched.step()

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch in val_dl:
                loss, _, _ = compute_losses(model, batch, device)
                val_loss += loss.item()

        train_loss /= max(len(train_dl), 1)
        val_loss /= max(len(val_dl), 1)
        print(f"epoch {epoch+1}/{epochs}  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}")

        if val_loss < best_val:
            best_val = val_loss
            torch.save(model.state_dict(), out_path)

    print("training done, best val loss:", best_val)



In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

train(
        train_dir="output",
        val_dir="output",
        epochs=20,
        batch_size=14,
        lr=1e-3,
        device="cuda",
    )


device: cuda


In [ ]:

from __future__ import annotations

import json
import os
import time

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch

from inference import localize, TOL_PX
from model import SiamNCCLocalizer


def run(test_dir="data/test", checkpoint="driftsense_v2.pt", device="cuda", show=True, out_dir=None):
    device = torch.device(device if torch.cuda.is_available() else "cpu")
    model = SiamNCCLocalizer().to(device)
    model.load_state_dict(torch.load(checkpoint, map_location=device))
    model.eval()

    with open(os.path.join(test_dir, "ground_truth.json")) as f:
        records = json.load(f)

    results = []
    for rec in records[:10]:
        pid = rec["pair_id"]
        ref = cv2.imread(os.path.join(test_dir, "reference", f"pair_{pid:04d}_ref.png"), cv2.IMREAD_GRAYSCALE)
        search = cv2.imread(os.path.join(test_dir, "search", f"pair_{pid:04d}_search.png"), cv2.IMREAD_GRAYSCALE)
        t0 = time.perf_counter()
        out = localize(model, ref, search, device)
        dt = time.perf_counter() - t0
        err = float(np.hypot(out["x"] - rec["center_x_wide_px"], out["y"] - rec["center_y_wide_px"]))
        results.append(dict(pair_id=pid, error_px=err, time_sec=dt,
                             hard_case=rec.get("hard_case", False),
                             n_peaks=out["n_peaks_total"],
                             n_ambiguous=out["n_distinct_ambiguous_matches"]))

    errs = np.array([r["error_px"] for r in results])
    times = np.array([r["time_sec"] for r in results])
    hard_mask = np.array([r["hard_case"] for r in results])
    n_amb = np.array([r["n_ambiguous"] for r in results])

    def maybe_save(fig, name):
        if out_dir:
            os.makedirs(out_dir, exist_ok=True)
            fig.savefig(os.path.join(out_dir, name), dpi=140, bbox_inches="tight")

    # ---- Error histogram ----
    fig = plt.figure(figsize=(6, 4))
    plt.hist(errs, bins=20)
    plt.axvline(TOL_PX, color="red", linestyle="--", label=f"tolerance ({TOL_PX}px)")
    plt.xlabel("localization error (px)"); plt.ylabel("count")
    plt.title("Error distribution"); plt.legend()
    maybe_save(fig, "error_hist.png")
    if show: plt.show()
    else: plt.close(fig)

    # ---- Success rate: normal vs hard cases ----
    normal_sr = np.mean(errs[~hard_mask] <= TOL_PX) * 100 if (~hard_mask).any() else 0
    hard_sr = np.mean(errs[hard_mask] <= TOL_PX) * 100 if hard_mask.any() else 0
    fig = plt.figure(figsize=(4, 4))
    plt.bar(["normal", "hard"], [normal_sr, hard_sr], color=["steelblue", "indianred"])
    plt.ylabel("success rate (%)"); plt.ylim(0, 100)
    plt.title(f"Success rate @ {TOL_PX}px tolerance")
    for i, v in enumerate([normal_sr, hard_sr]):
        plt.text(i, v + 2, f"{v:.1f}%", ha="center")
    maybe_save(fig, "success_rate.png")
    if show: plt.show()
    else: plt.close(fig)

    # ---- Inference time distribution ----
    fig = plt.figure(figsize=(6, 4))
    plt.hist(times, bins=20, color="seagreen")
    plt.xlabel("inference time (sec)"); plt.ylabel("count")
    plt.title("Per-pair inference time")
    maybe_save(fig, "inference_time.png")
    if show: plt.show()
    else: plt.close(fig)

    # ---- Ambiguity vs error: is periodic aliasing actually the failure driver? ----
    fig = plt.figure(figsize=(5, 4))
    plt.scatter(n_amb, errs, alpha=0.4, s=14)
    plt.axhline(TOL_PX, color="red", linestyle="--", linewidth=1)
    plt.xlabel("# distinct near-tied peaks (ambiguity)"); plt.ylabel("error (px)")
    plt.title("Error vs. periodic-match ambiguity")
    maybe_save(fig, "ambiguity_vs_error.png")
    if show: plt.show()
    else: plt.close(fig)

    best_idx = int(np.argmin(errs))
    rec = records[best_idx]
    pid = rec["pair_id"]
    ref = cv2.imread(os.path.join(test_dir, "reference", f"pair_{pid:04d}_ref.png"), cv2.IMREAD_GRAYSCALE)
    search = cv2.imread(os.path.join(test_dir, "search", f"pair_{pid:04d}_search.png"), cv2.IMREAD_GRAYSCALE)
    out = localize(model, ref, search, device)
    fig, axs = plt.subplots(1, 3, figsize=(15, 5))
    axs[0].imshow(ref, cmap="gray"); axs[0].set_title("Reference"); axs[0].axis("off")

    axs[1].imshow(search, cmap="gray"); axs[1].set_title("Search + prediction vs GT (worst case)")
    gx0, gy0, gx1, gy1 = rec["bbox_wide_px"]
    axs[1].add_patch(plt.Rectangle((gx0, gy0), gx1 - gx0, gy1 - gy0, edgecolor="lime", facecolor="none", lw=2, label="GT"))
    axs[1].scatter([out["x"]], [out["y"]], c="red", marker="x", s=100, label="predicted")
    axs[1].legend(); axs[1].axis("off")

    axs[2].imshow(out["heatmap"], cmap="hot")
    axs[2].set_title(f"Predicted heatmap ({out['n_peaks_total']} peaks, "
                      f"{out['n_distinct_ambiguous_matches']} distinct)")
    axs[2].axis("off")
    plt.tight_layout()
    maybe_save(fig, "best_case_panel.png")
    if show: plt.show()
    else: plt.close(fig)


    # ---- Visual sanity check: pick the worst failure, not always records[0],
    # since a random/first-index example won't reliably show a failure mode ----
    worst_idx = int(np.argmax(errs))
    rec = records[worst_idx]
    pid = rec["pair_id"]
    ref = cv2.imread(os.path.join(test_dir, "reference", f"pair_{pid:04d}_ref.png"), cv2.IMREAD_GRAYSCALE)
    search = cv2.imread(os.path.join(test_dir, "search", f"pair_{pid:04d}_search.png"), cv2.IMREAD_GRAYSCALE)
    out = localize(model, ref, search, device)

    fig, axs = plt.subplots(1, 3, figsize=(15, 5))
    axs[0].imshow(ref, cmap="gray"); axs[0].set_title("Reference"); axs[0].axis("off")

    axs[1].imshow(search, cmap="gray"); axs[1].set_title("Search + prediction vs GT (worst case)")
    gx0, gy0, gx1, gy1 = rec["bbox_wide_px"]
    axs[1].add_patch(plt.Rectangle((gx0, gy0), gx1 - gx0, gy1 - gy0, edgecolor="lime", facecolor="none", lw=2, label="GT"))
    axs[1].scatter([out["x"]], [out["y"]], c="red", marker="x", s=100, label="predicted")
    axs[1].legend(); axs[1].axis("off")

    axs[2].imshow(out["heatmap"], cmap="hot")
    axs[2].set_title(f"Predicted heatmap ({out['n_peaks_total']} peaks, "
                      f"{out['n_distinct_ambiguous_matches']} distinct)")
    axs[2].axis("off")
    plt.tight_layout()
    maybe_save(fig, "worst_case_panel.png")
    if show: plt.show()
    else: plt.close(fig)

    print(f"summary: n={len(results)}  mean_err={errs.mean():.2f}px  median_err={np.median(errs):.2f}px  "
          f"success@{TOL_PX}px={100*np.mean(errs<=TOL_PX):.1f}%  mean_time={times.mean()*1000:.1f}ms  "
          f"pct_ambiguous={100*np.mean(n_amb>1):.1f}%")
    return results



   

In [ ]:
run("output","driftsense_v2.pt","cuda",True,"run/output")